In [ ]:
# ===== THƯ VIỆN (giống code mẫu) =====
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------------------------------------
# 1. THỰC NGHIỆM CÁC GIÁ TRỊ K (dữ liệu CHƯA chuẩn hóa)
# ---------------------------------------------------------
k_values = range(1, 21)
acc_raw = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)   # k = số hàng xóm gần nhất
    knn.fit(X_train, y_train)
    acc_raw.append(accuracy_score(y_test, knn.predict(X_test)))

best_k_raw = k_values[int(np.argmax(acc_raw))]  # K cho độ chính xác cao nhất (K đầu tiên nếu bằng nhau)
print(f"[Chưa chuẩn hóa] K tối ưu = {best_k_raw}, accuracy = {max(acc_raw):.3f}")

# ---------------------------------------------------------
# 2. CHUẨN HÓA DỮ LIỆU (StandardScaler)
# ---------------------------------------------------------
# KNN dựa trên khoảng cách nên đặc trưng có thang đo lớn sẽ lấn át đặc trưng nhỏ.
# Chuẩn hóa đưa mọi đặc trưng về mean=0, std=1.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit CHỈ trên tập train
X_test_s = scaler.transform(X_test)        # tập test chỉ transform (tránh rò rỉ dữ liệu)

acc_scaled = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    acc_scaled.append(accuracy_score(y_test, knn.predict(X_test_s)))

best_k = k_values[int(np.argmax(acc_scaled))]
print(f"[Đã chuẩn hóa]   K tối ưu = {best_k}, accuracy = {max(acc_scaled):.3f}")

# Vẽ so sánh accuracy theo K
plt.plot(k_values, acc_raw, marker="o", label="Chưa chuẩn hóa")
plt.plot(k_values, acc_scaled, marker="s", label="Đã chuẩn hóa")
plt.title("Accuracy theo giá trị K")
plt.xlabel("K")
plt.ylabel("Accuracy")
plt.xticks(k_values)
plt.legend()
plt.show()

# ---------------------------------------------------------
# 3. THỰC NGHIỆM CÁC LOẠI KHOẢNG CÁCH (dùng K tối ưu, dữ liệu đã chuẩn hóa)
# ---------------------------------------------------------
# euclidean: sqrt(sum((a-b)^2))        (Minkowski với p=2)
# manhattan: sum(|a-b|)                (Minkowski với p=1)
# minkowski: (sum(|a-b|^p))^(1/p), thử thêm p=3
print("\nSo sánh khoảng cách (K =", best_k, ")")
configs = [
    ("Euclidean",       dict(metric="euclidean")),
    ("Manhattan",       dict(metric="manhattan")),
    ("Minkowski (p=3)", dict(metric="minkowski", p=3)),
]
for ten, params in configs:
    knn = KNeighborsClassifier(n_neighbors=best_k, **params)
    knn.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_s))
    print(f"{ten:<16}: accuracy = {acc:.3f}")

# NHẬN XÉT (điền theo kết quả thực tế khi chạy):
# - Iris khá dễ nên nhiều giá trị K cho accuracy bằng nhau; test chỉ có 30 mẫu
#   nên mỗi mẫu sai làm accuracy giảm ~0.033.
# - K quá nhỏ (K=1) dễ bị nhiễu (overfit); K quá lớn làm ranh giới giữa các lớp bị mờ.
# - Chuẩn hóa quan trọng hơn với dữ liệu có thang đo chênh lệch lớn.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------------------------------------
# 1. Tải dữ liệu giá nhà
# ---------------------------------------------------------
data = fetch_california_housing(as_frame=True)
df = data.frame.sample(n=5000, random_state=42)   # lấy 5000 mẫu cho nhanh
X = df.drop(columns="MedHouseVal")                # đặc trưng đầu vào
y = df["MedHouseVal"]                             # giá nhà (đơn vị 100.000 USD)

# ---------------------------------------------------------
# 2. Chia tập huấn luyện / kiểm tra
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------------------------------------
# 3. Chuẩn hóa: đưa các đặc trưng về cùng thang đo
# ---------------------------------------------------------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ---------------------------------------------------------
# 4 + 5. KNN Regressor với nhiều K, tính MSE và R2
# ---------------------------------------------------------
# Hồi quy KNN: giá dự đoán = TRUNG BÌNH giá của K hàng xóm gần nhất
print(f"{'K':<4}{'MSE':<10}{'R2'}")
for k in [3, 5, 7, 9]:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    y_pred = knn.predict(X_test_s)
    print(f"{k:<4}{mean_squared_error(y_test, y_pred):<10.4f}{r2_score(y_test, y_pred):.4f}")

# NHẬN XÉT: MSE càng nhỏ và R2 càng gần 1 thì mô hình càng tốt.
# Chọn K cho MSE nhỏ nhất / R2 lớn nhất trong bảng kết quả thực tế.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target

# ---------------------------------------------------------
# So sánh các cách chia: tỉ lệ test khác nhau (90/10, 80/20, 70/30, 60/40, 50/50)
# ---------------------------------------------------------
print("Tỉ lệ test | Accuracy")
for ts in [0.1, 0.2, 0.3, 0.4, 0.5]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    svm = SVC(kernel="rbf", C=1, gamma="scale")   # giống code mẫu của đề cương
    svm.fit(X_train, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test))
    print(f"{ts:<11}| {acc:.3f}")

# ---------------------------------------------------------
# Cùng tỉ lệ 80/20 nhưng đổi random_state, và dùng stratify
# ---------------------------------------------------------
# random_state khác -> các mẫu rơi vào tập test khác nhau -> accuracy dao động
# stratify=y giữ nguyên tỉ lệ 3 loài hoa trong cả train và test
print("\nrandom_state | Không stratify | Stratify")
for rs in [0, 1, 7, 42, 100]:
    row = []
    for strat in [None, y]:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=rs, stratify=strat)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)
        svm = SVC(kernel="rbf", C=1, gamma="scale").fit(X_train, y_train)
        row.append(accuracy_score(y_test, svm.predict(X_test)))
    print(f"{rs:<13}| {row[0]:<15.3f}| {row[1]:.3f}")

# NHẬN XÉT (điền theo kết quả thực tế):
# - Tập test nhỏ (10%) thì accuracy kém ổn định vì chỉ có 15 mẫu.
# - Tập train nhỏ (test 50%) có thể làm accuracy giảm nhẹ do mô hình ít dữ liệu học.
# - Đổi random_state làm kết quả dao động, nên đánh giá bằng 1 lần chia là chưa chắc chắn.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# --- Tải, chia, chuẩn hóa giống Bài 2 (cùng random_state nên cùng tập dữ liệu) ---
data = fetch_california_housing(as_frame=True)
df = data.frame.sample(n=5000, random_state=42)
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ---------------------------------------------------------
# SVR với kernel linear và rbf
# ---------------------------------------------------------
# SVR = SVM dạng hồi quy: tìm hàm dự đoán sao cho phần lớn điểm nằm trong
# "ống" epsilon quanh hàm đó. (Với bài toán phân lớp thì dùng SVC.)
# linear: đường/mặt phẳng tuyến tính; rbf: đường cong phi tuyến
# C: mức phạt sai số (lớn -> khớp dữ liệu chặt hơn)
print(f"{'Kernel':<8}{'MSE':<10}{'R2'}")
for kernel in ["linear", "rbf"]:
    svr = SVR(kernel=kernel, C=1)
    svr.fit(X_train_s, y_train)
    y_pred = svr.predict(X_test_s)
    print(f"{kernel:<8}{mean_squared_error(y_test, y_pred):<10.4f}{r2_score(y_test, y_pred):.4f}")

# NHẬN XÉT (điền theo kết quả thực tế):
# - Dữ liệu giá nhà có quan hệ phi tuyến nên rbf thường tốt hơn linear.
# - So sánh thêm với KNN Regressor ở Bài 2 để thấy mô hình nào phù hợp hơn.